#### **Step 6 - The Pipeline**

In [3]:
def answer_question(retriever, model, enc, device, eos_id, max_seq_len, question, k=3, method="hybrid", max_context_tokens=400):

    from generate import ask

    # search our chunks depending on the method specified 
    if method == "sparse":
        results = retriever.search_sparse(question, k=k)
    elif method == "dense":
        results = retriever.search_dense(question, k=k)
    elif method == "hybrid":
        results = retriever.search_hybrid(question, k=k)
    
    # the search will return a list of tuples of the top chunks - iterate over this list of tuples and extract the text from the chunks 
    # we will then return a string of chunk texts separated by a new line for each chunk
    chunk_texts = [chunk["text"] for chunk, _score in results]
    context = "\n".join(chunk_texts)

    # defensive: even with sensible chunk_size, joining several chunks could occasionally push context too long. 
    # Truncate defensively by token count before it ever reaches ask().
    # We will encode our text into tokens to see if its sie exceeds the token size context window limit 
    # we then truncate as needed and convert back into text after truncating the text so its tokenised vsriant fits the context window size
    context_ids = enc.encode(context) 
    if len(context_ids) > max_context_tokens: 
        context_ids = context_ids[:max_context_tokens] # truncate defensively just in case it exceeds the content limit
        context = enc.decode(context_ids)
    
    # query our transformer model using the context and question
    # remember our prompt being fed into the model is structured as f"Context: {context}\nQuestion: {question}\nAnswer:"
    answer = ask(model, enc, device, eos_id, max_seq_len, context, question)
    
    # return the answer, the context and the chunks for evaluation 
    return {
        "answer": answer,
        "context_used": context,
        "retrieved_chunks": results,
        }

**`answer_question` (Pipeline) Intuitive Summary**

**The Big Picture:** This is the piece that finally connects retrieval to generation — everything built so far (sparse, dense, hybrid, fusion) only answers "which chunks are relevant?" This function takes that answer, turns it into a prompt, and asks the fine-tuned model to actually generate a response.

- **Retrieve**
    - Given a `method` ("sparse", "dense", or "hybrid"), call the matching method on the already-built `retriever` to get the top-k `(chunk_dict, score)` tuples for the question
    - The retriever is passed in already built, not constructed here — building it is expensive (embedding the whole corpus), so it's done once outside this function and reused across every call

- **Build the context string**
    - Pull just the text out of each retrieved chunk, discarding the scores
    - Join those chunk texts together with newlines into one combined `context` string

- **Defensive truncation**
    - Even though each individual chunk was sized safely back in chunking, joining several chunks together can still push the combined context past a safe token budget
    - Encode `context` into tokens, and if it exceeds `max_context_tokens`, truncate the token list and decode back to text — this protects the prompt from silently overflowing the model's context window before it ever reaches `ask()`

- **Generate**
    - Build the prompt in the exact same template the model was fine-tuned on: `"Context: {context}\nQuestion: {question}\nAnswer:"`
    - Call `ask()` (the model's generation function) with this context and question to get the model's completion

- **Return, don't just print**
    - Return a dict containing the `answer`, the `context_used`, and the `retrieved_chunks` — not just the answer string alone
    - This makes the system inspectable: if an answer is wrong, you can check `retrieved_chunks` to see whether it was a *retrieval* failure (wrong chunks pulled) or a *generation* failure (right chunks, bad model output) — a distinction that's impossible to make if all you have is the final answer text

**Note: This function doesn't do any retrieval logic itself — it's a thin orchestration layer that calls the retriever, formats what comes back, and calls the model. All the real logic lives in the pieces built earlier (Parts 2-5); this just wires them together.**